# Tutorial 03: Handoff &amp; `as_tool` — Multi-Agent Coordination Patterns

**Series:** 02 — Agentic AI &amp; Tool-Calling Workflows  
**Tutorial:** 03-Handoff-and-as_tool  
**Level:** Intermediate  
**Estimated time:** 30–45 minutes  

---

## 🎯 What You Will Learn

By the end of this tutorial, you will be able to:

1. **Understand the full stack** — from raw `ChatCompletion` API tool-calling to the high-level Agent SDK.
2. **Implement handoff semantics** — transfer task responsibility and workflow state between agents.
3. **Build specialist sub-agents** — create single-purpose agents (billing, refund, finance, market).
4. **Route requests with raw API** — use `client.chat.completions.create()` for manual intent classification and routing.
5. **Convert agents into callable tools (`as_tool`)** — wrap an agent as a function tool for parallel invocation.
6. **Use `SQLiteSession`** — persist conversation state on disk and resume across sessions.
7. **Enable parallel tool execution** — `ModelSettings(parallel_tool_calls=True)` for concurrent sub-agent calls.

---

## 📋 Prerequisites

| Requirement | Details |
|---|---|
| **Conda environment** | `agentic_ai` (Python 3.13, see `agentic_ai_env_export.yml` in project root) |
| **API Key** | DeepSeek API key stored in `.env` as `DEEPSEEK_API_KEY` (see below) |
| **Core packages** | `openai>=1.40`, `openai-agents>=0.0.6`, `python-dotenv>=1.0` |
| **Kernel** | `agentic_ai` kernel (`C:\Users\ycasi\AppData\Roaming\jupyter\kernels\agentic_ai\kernel.json`) |
| **Previous tutorials** | `01-AsyncOpenAI-Agent-Basics.ipynb`, `02-WebSearch-Agent.ipynb` |

---

## 🔑 API Key: The `.env` File

This notebook reads your DeepSeek API key from a `.env` file located at:

```
tutorials/02-agentic-ai/.env
```

**The `.env` file must contain:**

```ini
DEEPSEEK_API_KEY=sk-your-deepseek-key-here
```

> ⚠️ **Important:** The `.env` file is **not committed to git** (it's in `.gitignore`).  
> The code uses `load_dotenv(override=True)` then `os.getenv("DEEPSEEK_API_KEY")` to read it.

To verify your key is correctly set, run the **Setup Verification** cell below (after the model lineup table).

---

## 🏗️ Architecture Overview

```mermaid
graph TB
    subgraph Config["🔑 Configuration"]
        ENV["📄 .env file<br/>DEEPSEEK_API_KEY=sk-..."]
    end

    subgraph LowLevel["📡 Low-Level: Raw ChatCompletion API"]
        CC["client.chat.completions.create()<br/>model='deepseek-v4-flash'"]
        TOOLS["tool_choice + tools[]<br/>manual routing &amp; parallel calls"]
    end

    subgraph HighLevel["🧠 High-Level: OpenAI Agents SDK"]
        AG["Agent(name, instructions, model)"]
        RUNNER["Runner.run(agent, input)"]
        HO["handoffs=[agent1, agent2]<br/>→ transfer responsibility"]
        AT["agent.as_tool()<br/>→ wrap as callable function"]
        MS["ModelSettings<br/>parallel_tool_calls=True"]
    end

    ENV --> CC
    ENV --> AG
    CC --> TOOLS
    AG --> RUNNER
    RUNNER --> HO
    RUNNER --> AT
    AT --> MS

    style Config fill:#f9f,stroke:#333
    style LowLevel fill:#bbf,stroke:#333
    style HighLevel fill:#bfb,stroke:#333
```

> **This tutorial covers BOTH layers.** We start with the raw API to understand *how* things work, then use the SDK for productivity.

---

## 🔧 Conda Environment Setup

```bash
# Activate the agentic_ai environment
conda activate agentic_ai

# Install required packages (if not already installed)
pip install -r tutorials/02-agentic-ai/requirements.txt
```

**Restart your Jupyter kernel** and select the `agentic_ai` kernel before running cells.

---

## 🧭 Tutorial Structure

| Section | Topic | API Level |
|---|---|---|
| **0** | DeepSeek API: Latest Model Lineup (v4-flash, v4-pro) | — |
| **1** | Environment Setup &amp; Verification | — |
| **2** | Concept: Handoff | 📡 Raw API → 🧠 SDK |
| **3** | Agent SDK: Triage Agent with Handoff | 🧠 SDK |
| **4** | Raw ChatCompletion API: Manual Routing &amp; Handoff | 📡 Raw API |
| **5** | Concept: `as_tool` | 📡 Raw API → 🧠 SDK |
| **6** | Raw ChatCompletion API: Parallel Tool Calls | 📡 Raw API |
| **7** | Agent SDK: Manager Agent with Parallel Sub-Agent Tools | 🧠 SDK |
| **8** | Summary &amp; Key Takeaways | — |

---

## 📖 Background: Why Multi-Agent Coordination?

### The Problem with Monolithic Agents

A single agent with a large, catch-all `instructions` prompt and many tools becomes:
- **Hard to debug** — which instruction caused the wrong behavior?
- **Slow to reason** — the LLM must parse a huge system prompt every turn.
- **Prone to tool confusion** — too many tools leads to incorrect tool selection.

### The Multi-Agent Solution

Instead, we **decompose** the problem into specialized sub-agents:

```mermaid
graph LR
    U["👤 User"] --> M["🎯 Manager/Triage Agent"]
    M -->|"handoff"| SA1["💳 Billing Agent"]
    M -->|"handoff"| SA2["💰 Refund Agent"]
    M -->|"as_tool"| SA3["📊 Finance Agent"]
    M -->|"as_tool"| SA4["📈 Market Agent"]

    SA1 --> R1["Final Answer"]
    SA2 --> R2["Final Answer"]
    SA3 --> M
    SA4 --> M
    M --> R3["Synthesized Answer"]

    style M fill:#ff9,stroke:#333
    style SA1 fill:#fcc,stroke:#333
    style SA2 fill:#fcc,stroke:#333
    style SA3 fill:#cfc,stroke:#333
    style SA4 fill:#cfc,stroke:#333
```

- **Handoff agents** (red) take over completely — they produce the final answer.
- **`as_tool` agents** (green) return results to the manager — the manager synthesizes.

> This notebook is part of the **02-agentic-ai** tutorial series under `tutorials/02-agentic-ai/`.  
> For original reference materials, see `assets/previous-resources/`.


---

## 📡 DeepSeek API — Latest Model Lineup (August 2026)

| Model | Description | Best For |
|---|---|---|
| `deepseek-v4-flash` | DeepSeek-V4-Flash-0731 — 1M context, tool calls ✅, JSON mode ✅ | Fast, cost-effective agent workflows (**used in this notebook**) |
| `deepseek-v4-pro` | DeepSeek-V4-Pro — 1M context, higher reasoning quality | Complex multi-step reasoning |

> **Base URL:** `https://api.deepseek.com` (OpenAI-compatible).  
> **New in V4:** Thinking mode with `reasoning_effort` parameter, 1M context window, peak/off-peak pricing.  
> **Legacy:** The old `deepseek-chat` alias still works but is deprecated — use `deepseek-v4-flash` for new code.

---

In [6]:
# =============================================================================
# CELL 1: Environment Setup — Load .env, Verify API Key, Initialize Client
# =============================================================================
# This cell:
#   1. Locates and loads the .env file (DEEPSEEK_API_KEY)
#   2. Masks and prints the key for verification (never the full key!)
#   3. Creates AsyncOpenAI client + OpenAIChatCompletionsModel bridge
#   4. Tests the setup with a simple ChatCompletion
#
# NOTE: deepseek-v4-flash has "thinking" mode ON by default.
# For simple tasks, we disable it with extra_body={"thinking": {"type": "disabled"}}.
# For agent workflows, the Agents SDK handles this automatically.
# =============================================================================

import os, asyncio
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI
from agents import (
    Agent, Runner, SQLiteSession,
    OpenAIChatCompletionsModel,
    set_tracing_disabled, ModelSettings, function_tool, RunContextWrapper
)
from pydantic import BaseModel

# ---------------------------------------------------------------------------
# 1. Locate and load the .env file
# ---------------------------------------------------------------------------
notebook_dir = Path.cwd()
env_path = notebook_dir / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f".env not found at {env_path}. "
        "Create a .env file with: DEEPSEEK_API_KEY=sk-your-key-here"
    )

load_dotenv(env_path, override=True)

api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("DEEPSEEK_API_KEY not found in .env file!")

# Mask the key for safe display
masked = api_key[:8] + "..." + api_key[-4:] if len(api_key) > 12 else "***"
print(f"✓ .env loaded from: {env_path}")
print(f"✓ DEEPSEEK_API_KEY: {masked}")
print()

# ---------------------------------------------------------------------------
# 2. Create the model bridge (DeepSeek via AsyncOpenAI)
# ---------------------------------------------------------------------------
# base_url: https://api.deepseek.com (no /v1 suffix, as of Aug 2026)
# deepseek-v4-flash has thinking mode ON by default — disable for simple tasks
set_tracing_disabled(True)

client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

# deepseek-v4-flash = DeepSeek-V4-Flash-0731 (latest, 1M context, tool calls ✓)
# Legacy alias "deepseek-chat" also maps to this model.
model = OpenAIChatCompletionsModel(
    model="deepseek-v4-flash",
    openai_client=client
)

# ---------------------------------------------------------------------------
# 3. Quick sanity check: sync ChatCompletion (disable thinking for short answers)
# ---------------------------------------------------------------------------
sync_client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

response = sync_client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role": "user", "content": "Say 'Handoff tutorial ready!' in 3 words."}
    ],
    max_tokens=30,
    temperature=0.0,
    extra_body={"thinking": {"type": "disabled"}},  # Disable thinking for quick responses
)

print(f"✓ API connection verified:")
print(f"  Model returned: {response.model}")
print(f"  Response text:  {response.choices[0].message.content!r}")
print(f"  Tokens used:    {response.usage.total_tokens}")

✓ .env loaded from: c:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\02-agentic-ai\.env
✓ DEEPSEEK_API_KEY: sk-007ed...32aa

✓ API connection verified:
  Model returned: deepseek-v4-flash
  Response text:  'Handoff tutorial ready.'
  Tokens used:    21


## Section 2: Concept — Handoff (Transferring Responsibility Between Agents)

### What Is a Handoff?

In multi-agent systems, a **handoff** is the act of one agent **transferring both task responsibility and conversation context** to another agent. Think of it like a customer service call center:

- You call the **general hotline** (triage agent).
- The operator listens and says: *"Let me transfer you to our billing department."*
- The **billing specialist** picks up, already knowing what you told the first operator.
- The first agent **does not return** — the receiving agent produces the final output.

### Sequence Diagram: Handoff Flow

```mermaid
sequenceDiagram
    actor User
    participant Triage as 🎯 Triage Agent
    participant Billing as 💳 Billing Agent
    participant Refund as 💰 Refund Agent

    User->>Triage: "I was charged twice this month!"
    activate Triage
    Triage->>Triage: Analyze intent → billing issue
    Triage-->>Billing: 🔄 Handoff + full conversation context
    deactivate Triage
    activate Billing
    Billing->>Billing: Process billing query with own tools
    Billing-->>User: ✅ Final billing resolution
    deactivate Billing

    Note over User,Refund: Alternative path — if user asked about refunds:
    User->>Triage: "I want a refund for my order"
    activate Triage
    Triage->>Triage: Analyze intent → refund issue
    Triage-->>Refund: 🔄 Handoff + full conversation context
    deactivate Triage
    activate Refund
    Refund-->>User: ✅ Final refund guidance
    deactivate Refund
```

### What Happens at the API Level?

Before we look at the SDK, let's understand what the **raw ChatCompletion API** does during a handoff:

```mermaid
sequenceDiagram
    participant Code as Python Code
    participant API as DeepSeek ChatCompletion API
    participant BillingLLM as Billing LLM

    Code->>API: POST /chat/completions<br/>{model: "deepseek-v4-flash",<br/>messages: [triage system + user query],<br/>tools: [transfer_to_billing, transfer_to_refund]}

    API-->>Code: {finish_reason: "tool_calls",<br/>tool_calls: [{name: "transfer_to_billing"}]}

    Note over Code: SDK detects handoff tool call,<br/>switches to billing agent.

    Code->>API: POST /chat/completions<br/>{model: "deepseek-v4-flash",<br/>messages: [billing system + full history]}

    API-->>Code: {finish_reason: "stop",<br/>content: "I can help with your billing..."}
```

> **Key insight:** Handoff is just a **tool call** where the "tool" is "transfer to another agent." The SDK handles the agent switching automatically.

### Key Rules of Handoff

| Rule | Explanation |
|---|---|
| **One at a time** | An agent can only hand off to **one agent per turn**. The `handoffs` list provides options, but only one is selected. |
| **Tools stay behind** | Tools are **not transferred** during handoff. Each agent keeps only the tools it was initialized with. |
| **Context is preserved** | The conversation history (user messages + previous agent outputs) is carried forward to the receiving agent. |
| **Sequential, not parallel** | Handoff is inherently sequential — you cannot hand off to two agents simultaneously. |
| **No return to caller** | Once handed off, the original agent does not resume. The receiving agent produces the final output. |

### When to Use Handoff vs `as_tool`

| Scenario | Use Handoff | Use `as_tool` |
|---|---|---|
| One specialist must handle the entire remainder of the task | ✅ | ❌ |
| Need results from multiple specialists, then synthesize | ❌ | ✅ |
| Specialists need different tool sets | ✅ | ✅ |
| Want parallel execution for speed | ❌ | ✅ |
| Simple routing to a single expert | ✅ | ❌ |

> **Mental model:** Handoff = *"You take over completely."* &nbsp;|&nbsp; `as_tool` = *"Give me your answer, then I'll continue."*


In [7]:
# =============================================================================
# Section 3: Agent SDK — Triage Agent with Handoff
# =============================================================================
# This cell uses the high-level OpenAI Agents SDK to implement handoff:
#   1. Create specialist agents (billing, refund) with narrow, focused roles.
#   2. Create a triage agent with a `handoffs` list pointing to specialists.
#   3. The LLM decides which specialist to hand off to — NO manual routing code.
#
# Under the hood, the SDK:
#   - Registers each agent in `handoffs` as a special "transfer" tool.
#   - When the LLM calls that tool, the SDK swaps agents + preserves context.
#   - The receiving agent produces the final answer.
#
# Required: DEEPSEEK_API_KEY in tutorials/02-agentic-ai/.env
# =============================================================================

import asyncio
import os
from dotenv import load_dotenv                     # Reads API keys from .env
from openai import AsyncOpenAI                     # Async (non-blocking) client
from agents import (                                # OpenAI Agents SDK
    Agent, Runner,
    OpenAIChatCompletionsModel, set_tracing_disabled
)

# ---------------------------------------------------------------------------
# Environment Setup
# ---------------------------------------------------------------------------
# load_dotenv reads DEEPSEEK_API_KEY from tutorials/02-agentic-ai/.env
# override=True: .env values beat any existing system env vars
# set_tracing_disabled(True): skip OpenAI's native tracing (not needed with DeepSeek)
load_dotenv(override=True)
set_tracing_disabled(True)

# AsyncOpenAI: non-blocking I/O for concurrent API calls
# base_url points to DeepSeek's OpenAI-compatible endpoint
# api_key pulled from .env via os.getenv("DEEPSEEK_API_KEY")
client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),          # ← From .env file
    base_url="https://api.deepseek.com"              # DeepSeek API endpoint (official OpenAI-compatible base)
)

# OpenAIChatCompletionsModel bridges the async client into the Agents SDK
model = OpenAIChatCompletionsModel(
    model="deepseek-v4-flash",                      # DeepSeek V4 Flash (latest, 2026-07-31)
    openai_client=client
)

# ---------------------------------------------------------------------------
# Define Specialist (Leaf) Agents
# ---------------------------------------------------------------------------
# These are terminal agents — they have NO handoffs. When invoked by the
# triage agent, they produce a final answer directly.
# Each has a narrow, domain-specific system prompt.

billing_agent = Agent(
    name="billing_agent",                           # ID used for handoff routing
    instructions=(
        "You are a billing expert. You handle questions about invoices, "
        "charges, payment methods, subscription plans, and billing cycles. "
        "Provide detailed, accurate, and helpful answers about any "
        "billing-related topics. If you don't know something, be honest."
    ),
    model=model
)

refund_agent = Agent(
    name="refund_agent",                            # ID used for handoff routing
    instructions=(
        "You are a refund expert. You handle questions about refund policies, "
        "refund eligibility, processing times, chargebacks, and disputes. "
        "Provide clear, step-by-step guidance on how to request and track "
        "refunds. Be empathetic and solution-oriented."
    ),
    model=model
)

# ---------------------------------------------------------------------------
# Define the Triage (Coordinator) Agent
# ---------------------------------------------------------------------------
# The triage agent itself has NO domain tools — its ONLY job is to route.
# The `handoffs` list gives the LLM a choice of specialists to transfer to.
# The instructions tell the LLM WHEN to use which specialist.

triage_agent = Agent(
    name="triage_agent",
    instructions=(
        "You are a customer service triage specialist. Your ONLY job is to "
        "route the user's request to the right department:\n"
        "- If the user asks about charges, invoices, payments, or "
        "subscriptions → hand off to billing_agent.\n"
        "- If the user asks about refunds, money-back, or disputes → hand "
        "off to refund_agent.\n"
        "- If the request doesn't match either, respond directly and ask "
        "for clarification.\n"
        "Do NOT try to answer billing/refund questions yourself."
    ),
    handoffs=[billing_agent, refund_agent],          # Available transfer targets
    model=model
)

# ---------------------------------------------------------------------------
# Run the Agent Workflow
# ---------------------------------------------------------------------------
# Runner.run() handles the full loop:
#   1. Send user message → triage agent's LLM
#   2. LLM returns: either text (respond directly) or tool_call (handoff)
#   3. If handoff → SDK switches to specialist, passes context, runs it
#   4. Specialist's final_output becomes result.final_output

async def main():
    """Run the triage agent — it should hand off to billing_agent."""
    result = await Runner.run(
        triage_agent,
        "I was charged twice for my subscription this month — "
        "can you help me figure out what happened?"
    )
    # result.final_output comes from whichever agent finished the task
    print(result.final_output)

# In Jupyter, top-level await works directly (no asyncio.run() needed)
await main()


I'm sorry to hear about the duplicate charge—that's frustrating, especially when you're just trying to keep track of your bills. I can definitely help you get to the bottom of this.

Let's start by gathering a few details so I can pinpoint what happened. Could you please provide:

1. **The account email or customer ID** associated with the subscription.
2. **The exact date(s)** of the duplicate charges and the **amounts** (including currency).
3. **Your payment method** (e.g., credit card, PayPal, etc.) and whether you see two separate transactions or one that was "pending" and later posted again.

---

While I look into the specifics, here are the **most common reasons** this occurs:

- **Payment retry after a temporary decline** – If your first payment attempt timed out or was flagged by the bank, the system may automatically retry, resulting in two successful captures.
- **Multiple subscriptions on the same account** – If you have more than one plan (e.g., a personal and a business 

## Section 4: Raw ChatCompletion API — Manual Routing &amp; Handoff

### Why Look Under the Hood?

Before the Agents SDK existed, developers implemented handoff **manually** using the raw ChatCompletion API. Understanding this helps you:

1. **Debug SDK behavior** — you'll know what the SDK is doing under the hood.
2. **Build custom orchestrators** — when the SDK is too rigid, you can build your own.
3. **Understand costs** — each handoff = an extra API call.

### Manual Handoff Pattern

```
┌──────────────────────────────────────────────────────────────┐
│                    Manual Handoff Flow                        │
│                                                              │
│  Step 1: Classify user intent                                │
│  ┌─────────────────────────────────────────────────────────┐ │
│  │ client.chat.completions.create(                         │ │
│  │   model="deepseek-chat",                                │ │
│  │   messages=[                                            │ │
│  │     {"role":"system","content":"Classify: billing or     │ │
│  │      refund or other"},                                 │ │
│  │     {"role":"user","content":"I was charged twice"}     │ │
│  │   ]                                                     │ │
│  │ ) → "billing"                                           │ │
│  └─────────────────────────────────────────────────────────┘ │
│                           │                                   │
│                           ▼                                   │
│  Step 2: Route to specialist system prompt                    │
│  ┌─────────────────────────────────────────────────────────┐ │
│  │ specialist = {"billing": BILLING_PROMPT,                │ │
│  │               "refund":  REFUND_PROMPT}                 │ │
│  │                                                         │ │
│  │ client.chat.completions.create(                         │ │
│  │   model="deepseek-chat",                                │ │
│  │   messages=[                                            │ │
│  │     {"role":"system","content":specialist["billing"]},  │ │
│  │     {"role":"user","content":"I was charged twice"}     │ │
│  │   ]                                                     │ │
│  │ ) → "This looks like a duplicate charge. Let me..."     │ │
│  └─────────────────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────────────┘
```

> **The Agents SDK does exactly this** — but wraps it in `Agent(handoffs=[...])` + `Runner.run()` so you don't write the routing logic.


In [8]:
# =============================================================================
# Section 4 (Code): Raw ChatCompletion API — Manual Handoff Implementation
# =============================================================================
# This cell demonstrates handoff WITHOUT the Agents SDK:
#   Step 1: Classify user intent using a ChatCompletion call.
#   Step 2: Route to the appropriate specialist system prompt.
#   Step 3: The specialist produces the final answer.
#
# NOTE: deepseek-v4-flash (Aug 2026) has "thinking" mode ON by default.
# For short/structured outputs, disable it with thinking: {type: "disabled"}.
# For longer responses, thinking is fine — just increase max_tokens accordingly.
# =============================================================================

import asyncio
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI

# ---- Same client setup as before ----
load_dotenv(override=True)

client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# ---------------------------------------------------------------------------
# Define specialist system prompts
# ---------------------------------------------------------------------------
BILLING_PROMPT = (
    "You are a billing expert. Handle questions about invoices, charges, "
    "payment methods, subscription plans, and billing cycles."
)

REFUND_PROMPT = (
    "You are a refund expert. Handle questions about refund policies, "
    "refund eligibility, processing times, chargebacks, and disputes."
)

SPECIALISTS = {"billing": BILLING_PROMPT, "refund": REFUND_PROMPT}

# ---------------------------------------------------------------------------
# Step 1: Classify intent
# ---------------------------------------------------------------------------
CLASSIFIER_PROMPT = (
    "Classify into billing, refund, or other. Output ONLY one word."
)

async def classify_intent(user_message: str) -> str:
    response = await client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": CLASSIFIER_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        max_tokens=20, temperature=0.0,
        extra_body={"thinking": {"type": "disabled"}},
    )
    intent = response.choices[0].message.content.strip().lower()
    return intent.split()[0] if intent else "other"

# ---------------------------------------------------------------------------
# Step 2: Route to specialist
# ---------------------------------------------------------------------------
async def route_to_specialist(intent: str, user_message: str) -> str:
    if intent not in SPECIALISTS:
        return f"Intent '{intent}' not supported. Try billing or refund."
    response = await client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": SPECIALISTS[intent]},
            {"role": "user",   "content": user_message},
        ],
        max_tokens=400,
        extra_body={"thinking": {"type": "disabled"}},  # v4-flash: disable thinking
    )
    return response.choices[0].message.content

# ---------------------------------------------------------------------------
# Step 3: Run pipeline
# ---------------------------------------------------------------------------
async def manual_handoff(user_message: str):
    print(f"User: {user_message}\n")
    intent = await classify_intent(user_message)
    print(f"→ Classified as: {intent}")
    answer = await route_to_specialist(intent, user_message)
    print(f"→ Specialist ({intent}) response:\n{answer}")

# Test: billing
await manual_handoff(
    "I was charged twice for my subscription this month — what happened?"
)

print("\n" + "-" * 50 + "\n")

# Test: refund
await manual_handoff(
    "I need a refund for a product I bought last week. How do I do that?"
)

User: I was charged twice for my subscription this month — what happened?

→ Classified as: billing
→ Specialist (billing) response:
I understand your concern about being charged twice for your subscription this month. Let me look into that for you right away.

To give you an accurate answer, could you please confirm a couple of details?

1. **Which subscription** was charged twice (e.g., Basic, Pro, or a specific product)?
2. **Do you have multiple payment methods** saved on your account (e.g., a credit card and a PayPal account)?

In the meantime, here are the two most common reasons for a double charge:

- **Two active subscriptions on the same account:** If you upgraded or downgraded recently, the system may have created a new plan while the old one was still active, resulting in two separate charges.
- **Payment authorization hold vs. actual charge:** Sometimes a payment provider places a temporary "authorization hold" that looks like a second charge. This hold is automatically re

## Section 5: Concept — `as_tool` (Wrapping Agents as Callable Functions)

### What Is `as_tool`?

The `Agent.as_tool()` method **converts an entire agent into a callable function tool** that other agents can invoke. Instead of *"you take over"* (handoff), the manager agent says: *"Run this sub-agent like a function, give me the result, and I'll keep going."*

### Sequence Diagram: `as_tool` with Parallel Execution

```mermaid
sequenceDiagram
    actor User
    participant Manager as 🎯 Manager Agent
    participant Finance as 📊 Finance Agent (as_tool)
    participant Market as 📈 Market Agent (as_tool)

    User->>Manager: "Analyze Tesla as an investment"
    activate Manager

    Note over Manager: LLM decides to call BOTH tools in parallel

    par Parallel tool calls
        Manager->>Finance: 🔧 get_financial_analysis("Tesla")
        activate Finance
        Finance->>Finance: Analyze financial health
        Finance-->>Manager: "Revenue: $96B, Net income: $15B..."
        deactivate Finance
    and
        Manager->>Market: 🔧 get_market_analysis("Tesla")
        activate Market
        Market->>Market: Analyze market position
        Market-->>Manager: "Market share: 22%, Key competitors..."
        deactivate Market
    end

    Note over Manager: Manager synthesizes both results

    Manager-->>User: ✅ Comprehensive investment thesis
    deactivate Manager
```

### Handoff vs `as_tool` — Detailed Comparison

| Aspect | Handoff | `as_tool` |
|---|---|---|
| **Control flow** | Transfer &amp; never return | Call &amp; wait for result |
| **Parallelism** | Sequential (one at a time) | Parallel (multiple at once) |
| **Use case** | "You take over from here" | "Run this and tell me what you found" |
| **Result ownership** | Sub-agent produces final output | Manager synthesizes the result |
| **Context** | Full conversation history passed | Sub-agent gets a fresh prompt |
| **Tool sharing** | Tools stay with each agent | Same — each sub-agent keeps its own tools |
| **Best for** | Single-specialist tasks, routing | Multi-perspective analysis, data gathering |

### How `as_tool` Works Under the Hood

When you call `agent.as_tool(tool_name="...", tool_description="...")`:

1. The SDK creates a **function tool** definition (JSON Schema) from the agent.
2. The tool's **name** and **description** are passed to the LLM's function-calling interface.
3. When the manager agent's LLM decides to call the tool, the SDK:
   - Creates a **new agent run** for the sub-agent.
   - Passes the tool arguments as the sub-agent's input.
   - Waits for the sub-agent to finish.
   - Returns the sub-agent's `final_output` as the tool's return value.
4. The manager agent then uses the returned value to continue reasoning or produce a final synthesis.

### `ModelSettings(parallel_tool_calls=True)`

This setting tells the LLM it can emit **multiple tool calls in a single response**. Without it:

```mermaid
graph LR
    LLM1[LLM Turn 1] -->|"call finance"| Wait1[Wait...] --> R1[Result] --> LLM2[LLM Turn 2]
    LLM2 -->|"call market"| Wait2[Wait...] --> R2[Result] --> Synth[Synthesize]
```

With `parallel_tool_calls=True`:

```mermaid
graph LR
    LLM[LLM Single Turn] -->|"call finance"| Wait[Wait for both...]
    LLM -->|"call market"| Wait --> R[Both Results] --> Synth[Synthesize]
```

> **~2× speedup** when tools are independent — both API calls happen concurrently.

### Decision Guide: Handoff or `as_tool`?

```mermaid
graph TD
    Q1{"Does ONE specialist need to handle everything?"}
    Q1 -->|Yes| Handoff["🔄 Use HANDOFF"]
    Q1 -->|No| Q2{"Do you need results from MULTIPLE specialists?"}
    Q2 -->|Yes| AsTool["🔧 Use as_tool<br/>+ parallel_tool_calls=True"]
    Q2 -->|No| Q3{"Is the manager just routing?"}
    Q3 -->|Yes| Handoff
    Q3 -->|No| AsTool

    style Handoff fill:#fcc,stroke:#333
    style AsTool fill:#cfc,stroke:#333
```


## Section 6: Raw ChatCompletion API — Parallel Tool Calls

### What the SDK Does for `as_tool`

Under the hood, `agent.as_tool()` generates a **JSON Schema function definition** and registers it with the ChatCompletion API. The manager LLM then uses **native tool calling** (`tool_choice`, `tools[]`) to invoke sub-agents. Let's see exactly how:

```mermaid
sequenceDiagram
    participant Code as Python Code
    participant API as DeepSeek ChatCompletion API

    Code->>API: POST /chat/completions<br/>{model: "deepseek-v4-flash",<br/>messages: [manager system + user],<br/>tools: [{name: "get_financial_analysis",...},<br/>{name: "get_market_analysis",...}]}

    API-->>Code: {finish_reason: "tool_calls",<br/>tool_calls: [<br/>  {name: "get_financial_analysis", args: {company:"Tesla"}},<br/>  {name: "get_market_analysis", args: {company:"Tesla"}}<br/>]}

    Note over Code: Execute BOTH tool functions.<br/>Note: In the raw API, WE execute the<br/>tools (not the SDK).

    Code->>Code: finance_result = analyze_finance("Tesla")
    Code->>Code: market_result = analyze_market("Tesla")

    Code->>API: POST /chat/completions<br/>{model: "deepseek-v4-flash",<br/>messages: [...,<br/>  {role:"tool", content: finance_result},<br/>  {role:"tool", content: market_result}]}

    API-->>Code: {finish_reason: "stop",<br/>content: "## Tesla Investment Analysis..."}
```

> **Key insight:** In the raw API, YOU write the tool execution loop. The SDK automates this with `Runner.run()`.


In [9]:
# =============================================================================
# Section 6 (Code): Raw ChatCompletion API — Parallel Tool Calls
# =============================================================================
# This cell implements parallel tool calls WITHOUT the Agents SDK.
# It shows exactly what the SDK's `as_tool` + `ModelSettings(parallel_tool_calls=True)`
# does under the hood:
#   1. Define tool schemas (JSON Schema function definitions).
#   2. Send them to the ChatCompletion API with the user's question.
#   3. The LLM decides to call both tools in PARALLEL (single response).
#   4. We execute both tools concurrently with asyncio.gather().
#   5. We send tool results back to the LLM for final synthesis.
#
# This is the "low-level" version of what Section 7 (Agent SDK) does.
# =============================================================================

import asyncio
import os
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv(override=True)

client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),           # From .env file
    base_url="https://api.deepseek.com"
)

MODEL = "deepseek-v4-flash"

# ---------------------------------------------------------------------------
# Step 1: Define tool schemas (JSON Schema format)
# ---------------------------------------------------------------------------
# These are the raw equivalent of Agent.as_tool().
# The LLM uses these schemas to decide when and how to call each tool.

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_financial_analysis",
            "description": (
                "Get a detailed financial health analysis for a given company. "
                "Covers revenue, profitability, debt, cash flow, and key ratios."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "company": {
                        "type": "string",
                        "description": "The company name (e.g. 'Tesla', 'Apple')"
                    }
                },
                "required": ["company"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_market_analysis",
            "description": (
                "Get a detailed market position analysis for a given company. "
                "Covers market share, competitors, industry trends, and growth."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "company": {
                        "type": "string",
                        "description": "The company name (e.g. 'Tesla', 'Apple')"
                    }
                },
                "required": ["company"]
            }
        }
    }
]

# ---------------------------------------------------------------------------
# Step 2: Define the actual function implementations
# ---------------------------------------------------------------------------
# In this demo, we use the LLM itself as the "tool implementation."
# In production, these would call real APIs, databases, or computation.
# This mirrors what Agent.as_tool() does — the sub-agent IS an LLM call.

async def get_financial_analysis(company: str) -> str:
    """Run a financial analysis sub-agent via ChatCompletion API."""
    response = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a senior financial analyst. Given a company name, "
                "analyze its financial health. Cover: revenue trends, "
                "profitability, debt levels, cash flow, and key ratios. "
                "Be quantitative where possible."
            )},
            {"role": "user", "content": f"Analyze the financial health of {company}."}
        ],
        max_tokens=400
    )
    return response.choices[0].message.content

async def get_market_analysis(company: str) -> str:
    """Run a market analysis sub-agent via ChatCompletion API."""
    response = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a senior market analyst. Given a company name, "
                "analyze its market position. Cover: market share, competitors, "
                "industry trends, growth opportunities, and key risks."
            )},
            {"role": "user", "content": f"Analyze the market position of {company}."}
        ],
        max_tokens=400
    )
    return response.choices[0].message.content

# Map tool names → async functions
TOOL_MAP = {
    "get_financial_analysis": get_financial_analysis,
    "get_market_analysis":   get_market_analysis,
}

# ---------------------------------------------------------------------------
# Step 3: The tool-calling loop (what Runner.run() does internally)
# ---------------------------------------------------------------------------

MANAGER_SYSTEM_PROMPT = (
    "You are a research manager. When given a company name, you MUST call "
    "BOTH get_financial_analysis AND get_market_analysis in parallel. "
    "After receiving both results, synthesize them into a comprehensive "
    "investment research summary covering: "
    "1) Financial health overview, "
    "2) Market position overview, "
    "3) Combined SWOT-style assessment, "
    "4) Overall investment thesis (bull case / bear case)."
)

async def run_with_parallel_tools(user_query: str):
    """Manual tool-calling loop with parallel execution."""
    messages = [
        {"role": "system", "content": MANAGER_SYSTEM_PROMPT},
        {"role": "user",   "content": user_query},
    ]

    # --- First API call: LLM decides which tools to call ---
    print("→ Sending request with tools to LLM...")
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"                             # LLM decides which tools
    )

    choice = response.choices[0]

    # If the LLM responds with text (no tool calls), return it.
    if choice.finish_reason == "stop":
        print("→ LLM answered directly (no tool calls):")
        return choice.message.content

    # If the LLM wants to call tools...
    if choice.finish_reason == "tool_calls":
        tool_calls = choice.message.tool_calls
        print(f"→ LLM requested {len(tool_calls)} tool call(s):")
        for tc in tool_calls:
            print(f"   - {tc.function.name}({tc.function.arguments})")

        # --- Execute ALL tool calls in PARALLEL using asyncio.gather ---
        async def execute_tool(tc):
            """Execute a single tool call and return the result message."""
            func_name = tc.function.name
            func_args = json.loads(tc.function.arguments)  # Parse JSON args
            result = await TOOL_MAP[func_name](**func_args)
            return {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            }

        # asyncio.gather runs all tool calls concurrently
        tool_results = await asyncio.gather(*[
            execute_tool(tc) for tc in tool_calls
        ])
        print(f"→ All {len(tool_results)} tool(s) executed in parallel")

        # --- Build the full message history ---
        # Append the assistant's tool_call message + all tool results
        messages.append(choice.message.model_dump())     # Assistant tool_calls
        messages.extend(tool_results)                     # Tool results

        # --- Second API call: LLM synthesizes the final answer ---
        print("→ Sending tool results back to LLM for synthesis...")
        final_response = await client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=800
        )
        return final_response.choices[0].message.content

# ---------------------------------------------------------------------------
# Run the parallel tool-calling workflow
# ---------------------------------------------------------------------------

result = await run_with_parallel_tools(
    "Analyze Tesla as an investment opportunity."
)
print("\n" + "=" * 60)
print("FINAL SYNTHESIZED ANSWER:")
print("=" * 60)
print(result)


→ Sending request with tools to LLM...
→ LLM requested 2 tool call(s):
   - get_financial_analysis({"company": "Tesla"})
   - get_market_analysis({"company": "Tesla"})
→ All 2 tool(s) executed in parallel
→ Sending tool results back to LLM for synthesis...

FINAL SYNTHESIZED ANSWER:
I notice the financial analysis returned empty. Let me retry that call to ensure we have complete data.

<｜｜DSML｜｜tool_calls>
<｜｜DSML｜｜invoke name="get_financial_analysis">
<｜｜DSML｜｜parameter name="company" string="true">Tesla</｜｜DSML｜｜parameter>
</｜｜DSML｜｜invoke>
</｜｜DSML｜｜tool_calls>


In [10]:
# =============================================================================
# Section 7: Agent SDK — Manager Agent with Parallel Sub-Agent Tools
# =============================================================================
# This is the high-level version of Section 6. Compare the two:
#   Section 6 (Raw API):   100+ lines, manual tool loop, manual asyncio.gather.
#   Section 7 (Agent SDK): ~50 lines, Agent.as_tool() + Runner.run() does it all.
#
# What the SDK does for us:
#   - Agent.as_tool() → generates JSON Schema tool definitions automatically.
#   - Runner.run()     → handles the tool-calling loop (send→tools→results→synthesize).
#   - ModelSettings(parallel_tool_calls=True) → tells LLM to emit parallel calls.
#   - SQLiteSession    → persists conversation to disk for audit/resumption.
#
# Required: DEEPSEEK_API_KEY in tutorials/02-agentic-ai/.env
# =============================================================================

import asyncio
import os
from dotenv import load_dotenv                              # Reads API keys from .env
from openai import AsyncOpenAI                              # Async client
from agents import (                                         # OpenAI Agents SDK
    Agent, Runner, SQLiteSession,                            # Core primitives
    OpenAIChatCompletionsModel, set_tracing_disabled,        # Model bridge
    ModelSettings                                            # LLM behavior tuning
)

# ---------------------------------------------------------------------------
# Environment Setup (same pattern as Section 3)
# ---------------------------------------------------------------------------
# Reads DEEPSEEK_API_KEY from tutorials/02-agentic-ai/.env
load_dotenv(override=True)
set_tracing_disabled(True)

client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),           # ← From .env file
    base_url="https://api.deepseek.com"
)

model = OpenAIChatCompletionsModel(
    model="deepseek-v4-flash",
    openai_client=client
)

# ---------------------------------------------------------------------------
# Define Specialist Sub-Agents
# ---------------------------------------------------------------------------
# These are the SAME "specialists" as in Section 6, but wrapped as Agent
# objects instead of raw functions. Each has its own system prompt and model.

finance_agent = Agent(
    name="Finance_Analyst",
    instructions=(
        "You are a senior financial analyst. Given a company name, analyze "
        "its financial health. Cover: revenue trends, profitability (gross "
        "margin, net income), debt levels, cash flow, and key financial "
        "ratios. Be quantitative where possible and note red flags or "
        "strengths."
    ),
    model=model
)

market_agent = Agent(
    name="Market_Analyst",
    instructions=(
        "You are a senior market analyst. Given a company name, analyze "
        "its market position. Cover: market share, competitive landscape, "
        "industry trends, growth opportunities, competitive moat, and key "
        "risks. Be specific about competitors and market dynamics."
    ),
    model=model
)

# ---------------------------------------------------------------------------
# Convert Sub-Agents to Tools using .as_tool()
# ---------------------------------------------------------------------------
# This SINGLE method call replaces ~30 lines of JSON Schema + function
# mapping from Section 6. The SDK:
#   - Extracts the agent's signature into a JSON Schema tool definition.
#   - Wires up the agent run as the tool's implementation.
#   - Handles error propagation and result formatting.

finance_tool = finance_agent.as_tool(
    tool_name="get_financial_analysis",              # ← LLM sees this name
    tool_description=(
        "Get a detailed financial health analysis for a given company. "
        "Covers revenue, profitability, debt, cash flow, and key ratios."
    )
)

market_tool = market_agent.as_tool(
    tool_name="get_market_analysis",                 # ← LLM sees this name
    tool_description=(
        "Get a detailed market position analysis for a given company. "
        "Covers market share, competitors, industry trends, and growth "
        "outlook."
    )
)

# ---------------------------------------------------------------------------
# Define the Manager (Coordinator) Agent
# ---------------------------------------------------------------------------
# tools=[finance_tool, market_tool] — these are FUNCTION tools, not handoffs.
# The manager calls them, gets results back, and synthesizes.
#
# ModelSettings(parallel_tool_calls=True) — this is the KEY setting.
# It tells the LLM: "You may emit MULTIPLE tool calls in a single response."
# Without it, tools are called one-at-a-time (sequential, ~2× slower).

manager_agent = Agent(
    name="Research_Manager",
    instructions=(
        "You are a research manager. When given a company name, you MUST "
        "call BOTH get_financial_analysis AND get_market_analysis in "
        "parallel. After receiving both results, synthesize them into a "
        "comprehensive investment research summary covering:\n"
        "1) Financial health overview\n"
        "2) Market position overview\n"
        "3) Combined SWOT-style assessment\n"
        "4) Overall investment thesis (bull case / bear case)"
    ),
    tools=[finance_tool, market_tool],               # Sub-agents as callable tools
    model=model,
    model_settings=ModelSettings(
        parallel_tool_calls=True                     # ⚡ Enable parallel execution
    )
)

# ---------------------------------------------------------------------------
# SQLiteSession: On-Disk Conversation Persistence
# ---------------------------------------------------------------------------
# SQLiteSession stores the full conversation to a local .db file.
# Benefits:
#   - Resume conversations across notebook restarts.
#   - Audit agent interactions after the fact.
#   - Share session state across multiple Runner.run() calls.
# The file `conversations.db` is created in the notebook's working directory.
session = SQLiteSession("demo_user", "conversations.db")

# ---------------------------------------------------------------------------
# Run the Agent Workflow
# ---------------------------------------------------------------------------
# Runner.run() handles everything:
#   → Send prompt to manager agent LLM
#   → LLM decides to call both tools in parallel
#   → SDK executes both sub-agents concurrently
#   → SDK feeds results back to manager LLM
#   → Manager synthesizes the final answer

async def main():
    """Run the manager to analyze a company with parallel sub-agent tools."""
    result = await Runner.run(
        manager_agent,
        "Analyze Tesla as an investment opportunity.",
        session=session                                # Persist to SQLite
    )
    # result.final_output = synthesized answer from the manager agent
    print(result.final_output)

await main()


I have the latest data from both analyses. Here is the comprehensive synthesized investment research summary.

---

# 📊 Tesla, Inc. (TSLA) — Comprehensive Investment Research Summary
*Based on FY2024 10-K and Q3 2024 earnings context*

---

## 1. Financial Health Overview

**Verdict: Solid but moderating — an elite balance sheet with a deteriorating income statement. The company remains solvent, profitable, and self-funding, but every operating trend is moving in the wrong direction.**

### Balance Sheet — The Economic Castle 💪
- **~$36.6B cash + investments** vs. ~$19.6B total debt (including customer financing facilities; core industrial debt is far lower) → **net cash positive, roughly +$17B**
- Debt-to-equity ~0.4x; current ratio ~1.7x — solid short-term liquidity
- No meaningful near-term refinancing risk; Tesla can fund its AI/robotaxi/energy ambitions entirely internally

### Revenue — Growth Has Stalled
| Segment | FY2024 | YoY |
|---------|--------|-----|
| **Total Revenue** |

## Section 8: Summary &amp; Key Takeaways

### What We Built — Full Architecture

```mermaid
graph TB
    subgraph Raw["📡 Raw ChatCompletion API (Sections 4 & 6)"]
        R1["Intent Classifier → Route → Specialist LLM"]
        R2["JSON Schema tools → Parallel asyncio.gather → Synthesize"]
    end

    subgraph SDK["🧠 Agent SDK (Sections 3 & 7)"]
        S1["Agent(handoffs=[...]) → Runner.run()"]
        S2["Agent.as_tool() + ModelSettings → Runner.run()"]
    end

    Raw -->|"automated by"| SDK

    style Raw fill:#bbf,stroke:#333
    style SDK fill:#bfb,stroke:#333
```

### Pattern Comparison

| | Raw ChatCompletion API | Agent SDK |
|---|---|---|
| **Handoff** | Manual classify → route to specialist prompt | `Agent(handoffs=[...])` + `Runner.run()` |
| **`as_tool`** | JSON Schema + manual `asyncio.gather` loop | `Agent.as_tool()` + `ModelSettings(parallel_tool_calls=True)` |
| **Lines of code** | ~100+ per pattern | ~30 per pattern |
| **Flexibility** | Full control over routing logic | SDK handles routing automatically |
| **Debugging** | You see every API call | SDK abstracts away the loop |

### Core Concepts Learned

| # | Concept | Raw API Equivalent | SDK Equivalent |
|---|---|---|---|
| 1 | **Decomposition** | Separate system prompts | `Agent(name, instructions)` |
| 2 | **Handoff** | Classify → re-call API with new prompt | `Agent(handoffs=[...])` |
| 3 | **`as_tool`** | JSON Schema + function mapping | `agent.as_tool(tool_name, tool_description)` |
| 4 | **Parallel execution** | `asyncio.gather(*tasks)` | `ModelSettings(parallel_tool_calls=True)` |
| 5 | **Persistence** | Manual DB writes | `SQLiteSession(user_id, db_path)` |
| 6 | **Model bridging** | `AsyncOpenAI(base_url=...)` | `OpenAIChatCompletionsModel(model, openai_client)` |

### SDK Building Blocks Reference

| Component | Purpose | Key Parameter(s) |
|---|---|---|
| `Agent` | Defines an AI persona with instructions, model, tools, and handoffs | `name`, `instructions`, `model`, `tools`, `handoffs` |
| `Agent.as_tool()` | Wraps an agent as a callable function tool for other agents | `tool_name`, `tool_description` |
| `Runner.run()` | Executes the full agent loop (LLM ⟷ tools) until completion | `agent`, `input`, `session` |
| `OpenAIChatCompletionsModel` | Adapts any OpenAI-compatible client for the Agents SDK | `model`, `openai_client` |
| `ModelSettings` | Tunes LLM behavior (parallel calls, temperature, etc.) | `parallel_tool_calls` |
| `SQLiteSession` | On-disk conversation persistence for resume/audit | `user_id`, `db_path` |
| `set_tracing_disabled()` | Disables OpenAI native tracing (use with third-party APIs) | `True`/`False` |

### Limitations &amp; Caveats

- **SDK is evolving** — APIs may change. Pin your `openai-agents` version.
- **Handoff is one-directional** — once transferred, no return path (without custom logic).
- **`as_tool` sub-agents are stateless** — each call = fresh run. Use `SQLiteSession` for state.
- **API costs** — each sub-agent call = separate API request. Monitor usage.
- **Parallel tools must be independent** — if tool B needs tool A's output, run sequentially.

### Where to Go Next

| Tutorial | What You'll Learn |
|---|---|
| `04-Replacement-01-MultiAgent.ipynb` | Guardrails, complex orchestration, nested handoffs |
| `01-AsyncOpenAI-Agent-Basics.ipynb` | Review fundamentals: Agent, Runner, model setup |
| `02-WebSearch-Agent.ipynb` | Building custom tools with `@function_tool` |

### Key Insight

> **The Agents SDK is a convenience layer over the ChatCompletion API.** Every feature — handoff, `as_tool`, parallel execution — can be implemented manually with `client.chat.completions.create()`. The SDK's value is in reducing boilerplate and handling edge cases. Understanding BOTH levels makes you a more effective AI engineer.


## Contact

- Business / HR: yucongcai_business@outlook.com
- Research: yucongcai_research@outlook.com


---

## Version log

| Version | Date | Change |
|---|---|---|
| v2.1 | 2026-08-04 | **API verification pass:** Updated model name `deepseek-chat` → `deepseek-v4-flash` (current DeepSeek-V4-Flash-0731), base URL `api.deepseek.com/v1` → `api.deepseek.com` per official docs. Added model lineup table (v4-flash vs v4-pro, thinking mode, 1M context). Verified against live DeepSeek API docs (2026-08-04). |
| v2.0 | 2026-08-04 | **Major restructure:** Added raw ChatCompletion API cells alongside Agent SDK — shows both "how it works" (raw API) and "how to use it" (SDK). Added Mermaid sequence diagrams (handoff flow, `as_tool` parallel flow, API-level interactions, decision tree). Added setup verification cell. |
| v1.1 | 2026-08-04 | Enhancement: detailed markdown explanations, architecture diagrams, comparison tables, thorough Python comments, conda environment notes, tutorial structure. |
| v1.0 | 2026-08-03 | Initial rebuild from `assets/previous-resources/` (archive kept untouched). |

### v2.1 changes (2026-08-04)

| Change |
|---|
| **Model name:** `deepseek-chat` → `deepseek-v4-flash` across all code cells (per official DeepSeek docs — `deepseek-v4-flash` = DeepSeek-V4-Flash-0731, 1M context) |
| **Base URL:** `https://api.deepseek.com/v1` → `https://api.deepseek.com` (official OpenAI-compatible base URL per docs) |
| **New cell:** DeepSeek API model lineup table (v4-flash vs v4-pro, features, pricing notes) |
| **Verified:** All changes checked against live https://api-docs.deepseek.com/ (2026-08-04) |

### v2.0 changes (2026-08-04)

| Change |
|---|
| Added raw ChatCompletion API manual handoff cell (classify → route to specialist) |
| Added raw ChatCompletion API parallel tool calls cell (JSON Schema + asyncio.gather) |
| Added 7 Mermaid diagrams across the notebook |
| Added explicit .env verification cell |
| Restructured into clear sections with Raw API ↔ SDK pairing |

### v1.1 changes (2026-08-04)

| Change |
|---|
| Added full tutorial header with objectives, prerequisites, conda env |
| Expanded concepts with diagrams and comparison tables |
| Enhanced code cells with detailed inline comments |

### v1.0 changes (2026-08-03)

| Change |
|---|
| No code changes needed (clean notebook) |
